# Milestone 3: Vanilla LSTM Encoder-Decoder — model check

Verifies `src/encoder.py` + `src/decoder.py` on a real batch from the
Milestone 2 pipeline. This is **not** a training run (that's Milestone 4) —
just proof that:

1. the architecture matches `docs/TODO.md`'s diagram (Embedding → LSTM
   Encoder → final hidden/cell → LSTM Decoder → Linear → vocab),
2. teacher forcing works during a training-mode forward pass and gradients
   flow end-to-end,
3. autoregressive decoding works when no target is given.

In [1]:
import sys
from pathlib import Path

import torch
import torch.nn as nn

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.normalize import normalize_en, normalize_ig
from src.vocab import Vocab
from src.dataset import load_pairs, encode_pairs, make_dataloader
from src.encoder import EncoderLSTM
from src.decoder import DecoderLSTM

DATA_DIR = PROJECT_ROOT / "data" / "benchmark_dataset"
MAX_LENGTH = 80  # from EDA/EDA.ipynb

## 1. Rebuild vocab + one real batch

Same steps as `EDA/EDA.ipynb`: build the train-split vocabularies, then
encode the train split into padded tensors and grab one batch.

In [2]:
train_pairs = load_pairs(
    DATA_DIR / "train_en.txt", DATA_DIR / "train_ig.txt", normalize_en, normalize_ig
)

en_vocab = Vocab("en")
ig_vocab = Vocab("ig")
for en, ig in train_pairs:
    en_vocab.add_sentence(en)
    ig_vocab.add_sentence(ig)

print(f"EN vocab: {en_vocab.n_words}, IG vocab: {ig_vocab.n_words}")

input_ids, target_ids = encode_pairs(train_pairs, en_vocab, ig_vocab, MAX_LENGTH)
train_loader = make_dataloader(input_ids, target_ids, batch_size=32, shuffle=True)

batch_input, batch_target = next(iter(train_loader))
print("batch_input :", batch_input.shape)
print("batch_target:", batch_target.shape)

EN vocab: 14215, IG vocab: 13580


batch_input : torch.Size([32, 80])
batch_target: torch.Size([32, 80])


## 2. Build the model

`hidden_size=256` is just a placeholder for this smoke test — Milestone 4
picks the real baseline value.

In [3]:
hidden_size = 256

encoder = EncoderLSTM(en_vocab.n_words, hidden_size)
decoder = DecoderLSTM(hidden_size, ig_vocab.n_words)

n_params = sum(p.numel() for p in encoder.parameters()) + sum(p.numel() for p in decoder.parameters())
print(f"Total parameters: {n_params:,}")

Total parameters: 11,658,252


## 3. Training-mode pass (teacher forcing)

Feed `batch_target` into the decoder so it's teacher-forced. Check the
output shape, then confirm the whole thing is differentiable: compute
`NLLLoss(ignore_index=Vocab.PAD_token)` and back-propagate. Padding is
ignored in the loss because most of each row past `<EOS>` is `<PAD>` —
without `ignore_index` the model would mostly learn to predict padding.

In [4]:
encoder_outputs, (enc_hidden, enc_cell) = encoder(batch_input)
decoder_outputs, (dec_hidden, dec_cell) = decoder(
    enc_hidden, enc_cell, MAX_LENGTH, target_tensor=batch_target
)

expected_shape = (batch_input.size(0), MAX_LENGTH, ig_vocab.n_words)
assert decoder_outputs.shape == expected_shape, decoder_outputs.shape
print("decoder_outputs shape:", tuple(decoder_outputs.shape), "OK")

criterion = nn.NLLLoss(ignore_index=Vocab.PAD_token)
loss = criterion(
    decoder_outputs.reshape(-1, decoder_outputs.size(-1)),
    batch_target.reshape(-1),
)
print("loss:", loss.item())

loss.backward()

enc_grad = encoder.embedding.weight.grad
dec_grad = decoder.out.weight.grad
assert enc_grad is not None and enc_grad.abs().sum().item() > 0, "no gradient reached encoder embedding"
assert dec_grad is not None and dec_grad.abs().sum().item() > 0, "no gradient reached decoder output layer"
print("Gradients flowed into both encoder and decoder. OK")

decoder_outputs shape: (32, 80, 13580) OK
loss: 9.528374671936035


Gradients flowed into both encoder and decoder. OK


## 4. Inference-mode pass (autoregressive)

Same batch, but `target_tensor=None` — the decoder must feed its own
predictions back in as the next input. The decoded sentence is untrained
gibberish (weights are random); the point is only that the mechanism runs
without a target and produces a well-formed id sequence.

In [5]:
encoder.eval()
decoder.eval()

with torch.no_grad():
    _, (enc_hidden, enc_cell) = encoder(batch_input)
    inference_outputs, _ = decoder(enc_hidden, enc_cell, MAX_LENGTH, target_tensor=None)

assert inference_outputs.shape == expected_shape, inference_outputs.shape
print("inference output shape:", tuple(inference_outputs.shape), "OK")

_, predicted_ids = inference_outputs.topk(1)
predicted_ids = predicted_ids.squeeze(-1)

sample_idx = 0
print("input (EN)    :", en_vocab.ids_to_sentence(batch_input[sample_idx]))
print("target (IG)   :", ig_vocab.ids_to_sentence(batch_target[sample_idx]))
print("predicted (IG):", ig_vocab.ids_to_sentence(predicted_ids[sample_idx]), "(untrained -- expected to be gibberish)")

inference output shape: (32, 80, 13580) OK
input (EN)    : locust beans
target (IG)   : ogiri ( ma ị chọọ )
predicted (IG): odion gbolagunte egbụ ajayi odion akporo tụbachara anyịnya ndịụka chebiriọdụ savage operation agbọghọ àlà kawa chukwu tetfund nwekwuo internally watford mechaala ịkwakpo iron utu nwepu ịkpeazụ imetu maxwell masoro onyeeze kalifonịa mbasa psg chebiriọdụ savage nime centre akwụrịrị nsọpụrụ emergency ụsoro gbadọsịrị ịkpazụ kwara fere goge efeela efeela ákwá 421 ebemaka azodo nnyefe causa ịmaliteghachi juziri enyi 10tama22 williams anụburum 217 enyeaka ọnwa incinerator states agbalịkwa monde mehiere nwaa katsina naịagha zulum ihedioha ịgbachị lilina ganduje owere uniben yighi gọọmenti (untrained -- expected to be gibberish)
